# 07 - Trend report + evaluation

Evaluate the LangGraph agent from notebook 06 against a single
LLM prompt that just gets the same current-window titles and is
asked to summarise. If the LangGraph output is not measurably
better, all that graph machinery is not earning its place.

**What you will learn**

- How to define and measure a trend detector's accuracy when the
  ground truth is a manual review, not a labelled dataset.
- How to tune the drill-down threshold: too low and every window
  triggers, too high and real spikes get missed.
- The trade-off between a single big LLM prompt (simple, opaque)
  and a multi-node LangGraph (modular, debuggable).
- How to present the final trend report in a form a human reader
  would actually want to read.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd

from langchain_core.messages import HumanMessage
from src.llm import get_chat_ollama
from src.trend import category_deltas

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 90)

## 1. Recap: the LangGraph report from nb06

Just print the saved report file. This is what the pipeline
produced on the split we used in nb06.

In [2]:
report_path = ROOT / "data" / "trend_report.md"
langgraph_report = report_path.read_text(encoding="utf-8")
print(langgraph_report)

# HN AI/ML trend report

- previous window: 111 stories
- current window:  112 stories

## Category share deltas (current vs previous)

- **research**: 23.2% (up +13.3 pp from 9.9%)
- **opinion**: 15.2% (down 10.9 pp from 26.1%)
- **tool**: 13.4% (up +3.5 pp from 9.9%)
- **product**: 22.3% (down 2.9 pp from 25.2%)
- **tutorial**: 8.9% (down 1.9 pp from 10.8%)
- **news**: 16.1% (down 1.0 pp from 17.1%)
- **other**: 0.9% (down 0.0 pp from 0.9%)

## Drill-down summaries

### research

The current trending topics on Hacker News in the 'research' category revolve around advancements and applications of Artificial Intelligence, particularly Large Language Models (LLMs). Several stories highlight the increasing portability and versatility of AI agents, with some even being used to create art, such as self-portraits. Researchers are also exploring ways to improve LLMs, including prompt caching and using quantum data for teaching. Additionally, there is a focus on verifying the authenticity of 

## 2. Manual verification of the top-N drilled categories

The drill-down node flagged categories whose share swung by
>= 3 pp. Read the actual current-window story titles inside each
flagged category and check whether the spike is real (there really
were more of these stories) or a labelling artefact (the LLM
tagged a bunch of unrelated stories with the same category).

In [3]:
stories = pd.read_csv(ROOT / "data" / "stories_ai.csv")
silver = pd.read_csv(ROOT / "data" / "silver_labels.csv")
merged = stories.merge(silver, on="id", how="inner")
merged["created_at"] = pd.to_datetime(merged["created_at"])
merged = merged.sort_values("created_at").reset_index(drop=True)
mid = len(merged) // 2
previous = merged.iloc[:mid].copy()
current = merged.iloc[mid:].copy()
print(f"previous window: {len(previous)} stories")
print(f"current  window: {len(current)} stories")

previous window: 111 stories
current  window: 112 stories


To check drill-down honesty we need the LLM-classifier
categories for the current window (same as nb06 used). Re-produce
them cheaply by loading nb06's saved trend report and parsing the
category rows. Where that is not enough we fall back to re-
classifying (small extra cost).

In [4]:
# nb06 saved the trend deltas inside the markdown report but not the
# per-row llm_category assignments. Fastest reproducible check: rerun
# the LLM classifier on the current window (same seed, same model).
from src.classify import LLMClassifier
chat = get_chat_ollama(model="llama3.1:8b")
clf = LLMClassifier(chat=chat)

titles = current["title"].fillna("").tolist()
texts = current["text"].fillna("").tolist()

t0 = time.time()
current["llm_category"] = clf.predict(titles, texts)
print(f"re-classified {len(current)} current-window stories in {time.time() - t0:.1f}s")

re-classified 112 current-window stories in 57.5s


In [5]:
# Recompute deltas exactly as nb06 did.
prev_cats = previous[["category"]].rename(columns={"category": "category"})
cur_cats = current[["llm_category"]].rename(columns={"llm_category": "category"})
trend = category_deltas(cur_cats, prev_cats, weight_col=None)
DRILL_THRESHOLD = 0.03
drill = trend[trend["delta"] >= DRILL_THRESHOLD]["category"].tolist()
print(f"categories that fired drill-down (>= {DRILL_THRESHOLD * 100:.0f}pp swing): {drill}")
print()
print(trend.to_string(index=False))

categories that fired drill-down (>= 3pp swing): ['research', 'tool']



category  current_share  previous_share     delta    ratio
research       0.232143        0.099099  0.133044 2.342532
 opinion       0.151786        0.261261 -0.109476 0.580973
    tool       0.133929        0.099099  0.034829 1.351461
 product       0.223214        0.252252 -0.029038 0.884885
tutorial       0.089286        0.108108 -0.018822 0.825893
    news       0.160714        0.171171 -0.010457 0.938910
   other       0.008929        0.009009 -0.000080 0.991071


### For each drilled category, show the titles

In [6]:
for cat in drill:
    subset = current[current["llm_category"] == cat]
    print(f"### {cat} ({len(subset)} current-window stories)")
    for t in subset["title"].tolist()[:15]:
        print(f"  - {t}")
    print()

### research (26 current-window stories)
  - AI Agents that are Portable
  - Prompt caching makes self-consistency cheap for long-context LLMs
  - Mathematics in the age of AI
  - I gave frontier LLMs a canvas and told them to draw a self-portrait
  - Teaching AI with Quantum Data
  - 100% RHAE on ARC-AGI-3 public with Claude Code, Opus 5, and one skill
  - Knowing your robot: the fiduciary program in the age of AI
  - AI Used to Verify Toughest Mathematics Proof Yet
  - Language has two parameters (ArXiv preprint)
  - Claude watermark- science behind it
  - Extensible Software in the age of LLMs
  - Can you tell which AI-generated text is watermarked?
  - A Constitution for One: 7 months governing a personal AI agent fleet
  - Stealing Reasoning Traces from Proprietary LLM APIs
  - Analysis of 3,602 ChatGPT ads: advertisers appeared in answers 8% of the time

### tool (15 current-window stories)
  - Show HN: Pack AI and scientific models as self-contained boxes
  - Sibbo – track what 

**Reading the titles.** Do they cohere? Two failure modes to
watch for:

- **Cluster contamination.** A category shows a spike because a
  handful of clearly-on-topic stories dominate. Legitimate.
- **Labeller drift.** A category shows a spike because the LLM
  classifier changed its mind about the fuzzy pairs (tutorial vs
  research, product vs tool) between the two windows. Then the
  spike is a labelling artefact, not a real trend.

If the titles in each drilled category are on-theme, the trend
detector is doing what it claims. If a drilled category is a
grab-bag of stories that do not share a real theme, the
detector-plus-labeller combination is producing false positives.

## 3. The single-prompt baseline

What would a naive summariser produce if we just handed the LLM
all current-window titles and asked for a summary of what is
trending? This is the comparison bar for the LangGraph agent.

In [7]:
def single_prompt_summary(titles: list[str]) -> str:
    joined = "\n".join(f"- {t}" for t in titles)
    prompt = (
        "You are producing a weekly summary of what is trending on "
        "Hacker News, focused on AI / ML stories. Read the story "
        "titles below and write a 3-6 sentence markdown paragraph "
        "describing the main themes and any specific stand-outs. "
        "Do not quote titles verbatim; describe the trends.\n\n"
        f"Titles:\n{joined}\n\nSummary:"
    )
    resp = chat.invoke([HumanMessage(content=prompt)])
    return resp.content.strip()


t0 = time.time()
baseline_report = single_prompt_summary(current["title"].fillna("").tolist())
print(f"single-prompt baseline generated in {time.time() - t0:.1f}s")
print()
print("--- baseline report ---")
print(baseline_report)

single-prompt baseline generated in 7.0s

--- baseline report ---
**AI/ML Trends on Hacker News**

This week's summary highlights the ongoing advancements in AI research, development, and applications. **Conversational AI** continues to be a major focus area, with discussions around meeting assistants, AI-powered chatbots, and the importance of privacy in these systems.

The trend of **LLMs (Large Language Models)** is also prominent, with several stories showcasing their capabilities, limitations, and potential uses. Researchers are exploring ways to improve LLMs' performance, such as prompt caching and self-consistency techniques.

**AI Safety and Ethics** are increasingly important topics, with discussions around watermarking AI-generated content, the need for transparency in AI decision-making, and the responsibility of developers to ensure their creations do not harm society.

The **intersection of AI and other fields**, like mathematics, quantum computing, and engineering, is als

## 4. Side-by-side

The two reports serve different framings:

- **Single-prompt** gives one flowing summary paragraph and does
  the whole job in a single LLM call. Fastest, simplest, cheapest.
- **LangGraph** gives per-category share deltas plus targeted
  drill-down summaries only on the categories that spiked.
  Slower (multiple LLM calls) but structured, debuggable, and
  easier to plug into a downstream alerting or dashboard system.

In [8]:
print("========================================================================")
print("SINGLE-PROMPT BASELINE")
print("========================================================================")
print(baseline_report)
print()
print("========================================================================")
print("LANGGRAPH AGENT REPORT")
print("========================================================================")
print(langgraph_report)

SINGLE-PROMPT BASELINE
**AI/ML Trends on Hacker News**

This week's summary highlights the ongoing advancements in AI research, development, and applications. **Conversational AI** continues to be a major focus area, with discussions around meeting assistants, AI-powered chatbots, and the importance of privacy in these systems.

The trend of **LLMs (Large Language Models)** is also prominent, with several stories showcasing their capabilities, limitations, and potential uses. Researchers are exploring ways to improve LLMs' performance, such as prompt caching and self-consistency techniques.

**AI Safety and Ethics** are increasingly important topics, with discussions around watermarking AI-generated content, the need for transparency in AI decision-making, and the responsibility of developers to ensure their creations do not harm society.

The **intersection of AI and other fields**, like mathematics, quantum computing, and engineering, is also gaining attention. Researchers are explor

**Reading the two.** Neither is universally "better"; the
LangGraph output is a *structured* view (numeric deltas + targeted
drill-downs) while the single-prompt is a *narrative* view. Real
production choice would depend on downstream consumer:

- If a human is reading it once a week: the single prompt is
  fine and probably preferred.
- If the output feeds a dashboard, an alerting rule ("notify me
  when research swings > 5pp"), or a follow-up automation:
  the LangGraph structure earns its keep because the numbers are
  in fields, not prose.

## 5. Latency and cost accounting

The LangGraph agent makes:
- N classifier calls (one per current-window story) in `node_classify`.
- 1 short LLM summary call per drilled category in `node_drill_down`
  (only fires conditionally).

The single-prompt baseline makes:
- 1 large prompt call.

For our N=112, roughly:

In [9]:
n_current = len(current)
n_drill = len(drill)
# Approximate token counts (very rough): classifier prompts ~300
# input tokens, ~5 output; drill-down ~600 input, ~120 output;
# baseline ~2000 input, ~300 output. Adjust to your model card.
langgraph_in  = n_current * 300 + n_drill * 600
langgraph_out = n_current *   5 + n_drill * 120
baseline_in   = 2000
baseline_out  = 300

print(f"Approximate token accounting for this window (N={n_current}, drilled={n_drill}):")
print(f"  LangGraph  in={langgraph_in:>6}  out={langgraph_out:>5}  total~{langgraph_in + langgraph_out}")
print(f"  Baseline   in={baseline_in:>6}  out={baseline_out:>5}  total~{baseline_in + baseline_out}")
ratio = (langgraph_in + langgraph_out) / (baseline_in + baseline_out)
print(f"  ratio: LangGraph is ~{ratio:.1f}x the tokens of the baseline")

Approximate token accounting for this window (N=112, drilled=2):
  LangGraph  in= 34800  out=  800  total~35600
  Baseline   in=  2000  out=  300  total~2300
  ratio: LangGraph is ~15.5x the tokens of the baseline


**Reading the ratio.** The LangGraph pipeline costs on the
order of tens of times more LLM tokens than the single-prompt
baseline (mostly from calling the classifier once per story). On
a local Ollama model this cost is time (a few minutes) not
dollars. On a hosted API it would be a small charge. In both
cases, if a single-prompt summary meets your needs, use it; the
LangGraph is worth it when the downstream needs the structured
categories + selective drill-downs.

## Wrapping up

- The full pipeline runs end-to-end: HN API -> qwen2.5:14b
  silver labels + Claude spot-check -> TF-IDF baseline
  (nb03, 43.3%) vs llama3.1:8b LLM classifier (nb04, 62.7%) ->
  sentence-transformer embeddings + KMeans / HDBSCAN (nb05) ->
  LangGraph agent with conditional drill-down (nb06) -> this
  final evaluation.
- Silver labels have a real ~62.5% agreement ceiling against a
  Claude review, so absolute-accuracy numbers below need that
  interpretation.
- The LangGraph agent produces measurably-different output from
  a single-prompt baseline; whether it is *better* depends on the
  downstream consumer (structured fields vs prose narrative).
- Category boundaries in `src/labels.py` are fuzzy on real HN
  content, especially the product / tool and tutorial / research
  pairs. A cleaner v2 might collapse them to a smaller,
  less-overlapping set.